# 多相机足球追踪系统 (MCS)

本笔记本提供项目中的一些最常用功能的说明

## 1. 系统概述

### 系统架构

```
┌─────────────────────────────────────────────────────────┐
│                   MultiCameraSystem (MCS)                │
└───────────────────────┬─────────────────────────────────┘
                        │
        ┌───────────────┼───────────────┐
        │               │               │
    ┌───▼────┐     ┌───▼────┐     ┌───▼────┐
    │Camera 1│     │Camera 2│     │Camera 3│
    └───┬────┘     └───┬────┘     └───┬────┘
        │              │              │
    ┌───▼──────────────▼──────────────▼───┐
    │  ● 相机标定 (Calibration)            │
    │  ● 目标检测 (YOLOX Detection)        │
    │  ● ReID 特征提取 (FastReID)          │
    │  ● 球场坐标转换 (Homography)         │
    └──────────────────┬──────────────────┘
                       │
    ┌──────────────────▼──────────────────┐
    │         MCSDetector                  │
    │  ● 多相机检测融合                     │
    │  ● 质量评分和过滤                     │
    │  ● Detection 缓存管理                 │
    │  ● 可视化输出                         │
    └─────────────────────────────────────┘
```

### 核心组件

| 组件 | 功能 | 文件 |
|------|------|------|
| **MultiCameraSystem** | 多相机系统管理 | `MCS.py` |
| **Camera** | 单相机管理（标定、检测） | `camera.py` |
| **Detection** | 检测对象（包含 ReID 特征） | `camera.py` |
| **MCSDetector** | 多相机检测融合 | `MCS.py` |
| **FastReIDInterface** | ReID 特征提取 | `BoT-SORT/fast_reid/` |

---

## 2. 环境设置

### 依赖包

```bash
# 创建 conda 环境
conda create -n football python=3.8
conda activate football

# 安装基础包
pip install -r requirements.txt

# 安装 YOLOX 和 BoT-SORT 依赖
# 
# BotSORT 拉取我fork的版本，改进中。根据BoTSORT文档中的步骤配置环境（需要编译）
cd BoT-SORT
pip install -r requirements.txt
```

### 预训练模型 (只使用标注功能不需要)

```bash
# YOLOX 检测模型
mkdir -p BoT-SORT/pretrained
# bytetrack_x_mot17.pth.tar

# 下载 FastReID 模型
mkdir -p pretrained
# mot17_sbs_S50.pth
```

### 目录结构

```
football_remote/
├── MCS.py                    # 主系统文件
├── camera.py                 # 相机类
├── config.py                 # 配置
├── soccer.py                 # 球场定义
├── BoT-SORT/                 # 追踪器
│   ├── fast_reid/           # ReID 模型
│   └── pretrained/          # YOLOX 权重
├── pretrained/               # FastReID 权重
├── videos/                   # 输入视频
└── store/                    # 输出结果
    └── <scenario_name>/
        ├── camera1/
        ├── camera2/
        ├── camera3/
        └── position/
            └── detections/   # Detection 缓存
```

---

## 3. 快速开始

### 最简单的使用流程

In [ ]:
# 导入必要的库
from soccer import *
from camera import Camera
from MCS import MCS

print("✅ 导入成功！")

✅ 导入成功！


- 关键点标注（用于单应转换）：尽量选取靠外侧的点（参考readme）4-6个即可。由于现在的场地数据，尽量不选取大小禁区的点

- 通用标注：分组标注，按“n”进入下一组
    - 横向标注               ：标注出画面中平行于侧边线的线（不一定非是球场的线）
    - 纵向标注（用于去畸变）  ：垂直于侧边线的线（比如草皮的线）
    - 训练区域（筛选检测目标）：标注出需要框定训练区域额外需要的点，结合场地关键点选出训练区域

In [4]:
# 创建球场配置
config = HalfWuChuanConfiguration()

video_path_1 = "videos/视角1.mp4"
video_path_2 = "videos/视角2.mp4"
video_path_3 = "videos/视角3.mp4"

# 初始化相机（标定）
camera_1 = Camera(video_path_1, config, camera_id=1, camera_name="camera1", mcs_name="wuchuan")
camera_2 = Camera(video_path_2, config, camera_id=2, camera_name="camera2", mcs_name="wuchuan")
camera_3 = Camera(video_path_3, config, camera_id=3, camera_name="camera3", mcs_name="wuchuan")


[Camera camera1] === 开始带畸变校正的标定流程 ===
[Camera camera1] 标定文件将保存至: store\wuchuan_mcs\camera1

[Camera camera1] 步骤 1/3: 球场关键点标定
[Camera camera1] 发现已有标定文件: store\wuchuan_mcs\camera1\keypoints.json
[Camera camera1] 加载成功。

[Camera camera1] 步骤 2/3: 畸变参数标定
[Camera camera1] 发现已有畸变参数: store\wuchuan_mcs\camera1\distortion_params.json
畸变参数已加载: store\wuchuan_mcs\camera1\distortion_params.json
[Camera camera1] 畸变参数加载成功。

[Camera camera1] 步骤 3/3: 计算单应矩阵（使用去畸变后的点）
[Transform] compute_transform_matrix called with 5 points.
[Transform] Homography matrix computed successfully.
[Camera camera1] 畸变校正标定完成！
[Camera camera1] 畸变参数: k1=-0.2088, k2=0.0274, k3=-0.1000
[FOV] left: 采样中间区域[540-1620], 过滤 21/100 个点, 保留 79 个
[FOV] left: 三次拟合完成, 输出 50 个点, Y范围 [-2124, 6486]
[FOV] right: 采样中间区域[540-1620], 过滤 24/100 个点, 保留 76 个
[FOV] right: 三次拟合完成, 输出 50 个点, Y范围 [-2166, 977]

[Camera camera1] === 可视化视频边界 ===
  视频分辨率: 3840x2160
  采样区域: 高度 540 到 1620 (避开上下边缘)
  采样点数: 100
  左边界球场坐标范围: X[1039, 8631] Y[-47806, 124963]
  右边界球场坐

Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.


[Camera camera1] FastReID initialized:
  - Config: D:\intern\teenager_football\football_remote\BoT-SORT\fast_reid\configs\MOT17\sbs_S50.yml
  - Weights: D:\intern\teenager_football\football_remote\BoT-SORT\pretrained\mot17_sbs_S50.pth
  - Device: cuda
  - Batch size: 8
[Camera camera1] Detection initialized with yolox model: D:\intern\teenager_football\football_remote\BoT-SORT\pretrained\bytetrack_x_mot17.pth.tar
[Camera camera1] ReID feature extraction enabled

[Camera camera2] === 开始带畸变校正的标定流程 ===
[Camera camera2] 标定文件将保存至: store\wuchuan_mcs\camera2

[Camera camera2] 步骤 1/3: 球场关键点标定
[Camera camera2] 发现已有标定文件: store\wuchuan_mcs\camera2\keypoints.json
[Camera camera2] 加载成功。

[Camera camera2] 步骤 2/3: 畸变参数标定
[Camera camera2] 发现已有畸变参数: store\wuchuan_mcs\camera2\distortion_params.json
畸变参数已加载: store\wuchuan_mcs\camera2\distortion_params.json
[Camera camera2] 畸变参数加载成功。

[Camera camera2] 步骤 3/3: 计算单应矩阵（使用去畸变后的点）
[Transform] compute_transform_matrix called with 10 points.
[Transform] Homograp

Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.


[Camera camera2] FastReID initialized:
  - Config: D:\intern\teenager_football\football_remote\BoT-SORT\fast_reid\configs\MOT17\sbs_S50.yml
  - Weights: D:\intern\teenager_football\football_remote\BoT-SORT\pretrained\mot17_sbs_S50.pth
  - Device: cuda
  - Batch size: 8
[Camera camera2] Detection initialized with yolox model: D:\intern\teenager_football\football_remote\BoT-SORT\pretrained\bytetrack_x_mot17.pth.tar
[Camera camera2] ReID feature extraction enabled

[Camera camera3] === 开始带畸变校正的标定流程 ===
[Camera camera3] 标定文件将保存至: store\wuchuan_mcs\camera3

[Camera camera3] 步骤 1/3: 球场关键点标定
[Camera camera3] 发现已有标定文件: store\wuchuan_mcs\camera3\keypoints.json
[Camera camera3] 加载成功。

[Camera camera3] 步骤 2/3: 畸变参数标定
[Camera camera3] 发现已有畸变参数: store\wuchuan_mcs\camera3\distortion_params.json
畸变参数已加载: store\wuchuan_mcs\camera3\distortion_params.json
[Camera camera3] 畸变参数加载成功。

[Camera camera3] 步骤 3/3: 计算单应矩阵（使用去畸变后的点）
[Transform] compute_transform_matrix called with 5 points.
[Transform] Homograph

Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.


[Camera camera3] FastReID initialized:
  - Config: D:\intern\teenager_football\football_remote\BoT-SORT\fast_reid\configs\MOT17\sbs_S50.yml
  - Weights: D:\intern\teenager_football\football_remote\BoT-SORT\pretrained\mot17_sbs_S50.pth
  - Device: cuda
  - Batch size: 8
[Camera camera3] Detection initialized with yolox model: D:\intern\teenager_football\football_remote\BoT-SORT\pretrained\bytetrack_x_mot17.pth.tar
[Camera camera3] ReID feature extraction enabled


In [7]:
# 多相机系统初始化
cameras = [camera_1, camera_2, camera_3]
mcs = MCS(cameras, config, name="wuchuan")

[MCS] 存储目录: store\wuchuan_mcs
[GridSystem] 初始化: 88行 × 66列 = 5808 个网格
[GridSystem] 球场: 3300cm × 4400cm, 网格: 50cm

[MCS] 计算网格质量分数（Brown-Conrady物理模型）
网格系统: 88行 × 66列

模型参数:
  距离分数 (Sigmoid):
    - 半衰点比例: 0.50
    - 陡峭度因子: 8.0
  畸变分数 (Brown-Conrady物理模型):
    - 畸变量缩放系数: 50.0
  组合方式: 乘法 (Q = S_dist × S_distortion)

相机畸变信息:
  相机 1:
    畸变系数: k1=-0.2088, k2=0.0274, k3=-0.1000
    图像尺寸: 3840×2160 px
    归一化半径: 2202.9 px
  相机 2:
    畸变系数: k1=-0.0280, k2=0.0360, k3=-0.0189
    图像尺寸: 3840×2160 px
    归一化半径: 2202.9 px
  相机 3:
    畸变系数: k1=-0.1664, k2=0.0500, k3=-0.0745
    图像尺寸: 3840×2160 px
    归一化半径: 2202.9 px

Computing scores for Camera 1...
  使用预计算的可见网格: 5186 个网格
  > Cam 1: 可见网格=5186, 有效分数=5073, 最大分数=0.9538
Computing scores for Camera 2...
  使用预计算的可见网格: 4646 个网格
  > Cam 2: 可见网格=4646, 有效分数=4645, 最大分数=0.9671
Computing scores for Camera 3...
  使用预计算的可见网格: 5803 个网格
  > Cam 3: 可见网格=5803, 有效分数=5650, 最大分数=0.9615

[MCS] 网格分数计算完成
  - 相机数: 3
  - 网格系统: 88行 × 66列

MCS: Score maps computed for 3 cameras

M

## 目标检测

使用 YOLOX 进行球员检测，并转换到球场坐标系统。

### 检测流程

```
视频帧 → YOLOX检测 → 坐标转换 → 范围过滤 → Detection对象
```

In [ ]:
# mcs系统进行目标检测
mcs.detect()
# mcs.detect_colors() 颜色检测，调用api
mcs.save_detections_cache(cache_name="with_reid") # 缓存检测目标信息

In [14]:
# camera
camera_1.detection.detect()
camera_1.detection.visualize_on_pitch(pitch_config=config)

[CameraDetection] Camera camera1: 成功提取 30 个ReID特征 (维度: (2048,))
[Camera camera1] ✅ 检测可视化已保存: store\wuchuan_mcs\camera1\detection\camera1_detection_pitch.jpg


array([[[ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34],
        ...,
        [ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34]],

       [[ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34],
        ...,
        [ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34]],

       [[ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34],
        ...,
        [ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34]],

       ...,

       [[ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34],
        ...,
        [ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34]],

       [[ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34],
        ...,
        [ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34]],

       [[ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34],
        ...,
        [ 34, 139,  34],
        [ 34, 139,  34],
        [ 34, 139,  34]]

## ReID 特征提取

ReID (Re-identification) 用于提取球员的外观特征，用于跨相机追踪。默认在detect()中开启

### ReID 工作流程

```
检测框 → 裁剪图像 → FastReID模型 → L2归一化 → 特征向量(2048维)
```

### ReID 特征用途

- **相似度计算**: `similarity = np.dot(feat1, feat2)`
- **身份匹配**: 跨帧、跨相机的球员关联
- **追踪优化**: 与 IoU 和位置信息融合

### 当前的追踪器

In [ ]:
from tools.single_camera_tracker import tracker_botsort

tracker = tracker_botsort(camera= camera_1)
tracker.save_tracking_video(refine= False)

## 球场配置可视化

`soccer.py` 中定义的所有球场配置类，可通过 `draw_pitch_with_keypoints` 直接渲染为带标注的俯视图。

- **粉红点**：禁区、底线区域关键点
- **天蓝点**：中线、中圈关键点
- **橙红点**：全场对侧禁区关键点
- **白色点**：辅助中轴点
- **绿色小点**：草皮分界点（T/B 前缀，自动缩小显示）

In [ ]:
import matplotlib.pyplot as plt
import cv2
from soccer import (
    SoccerPitchConfiguration, HalfPitchConfiguration,
    WuChuanConfiguration, HalfWuChuanConfiguration,
    QingYuanConfiguration, HalfQingYuanConfiguration,
    CUMTBPitchConfiguration,
)
from tools.annotation.pitch_annotation_tool import draw_pitch_with_keypoints

configs = {
    "SoccerPitchConfiguration\n(标准全场 120×70m)":    SoccerPitchConfiguration(),
    "HalfPitchConfiguration\n(标准半场 60×70m)":       HalfPitchConfiguration(),
    "WuChuanConfiguration\n(吴川全场 66×44m)":         WuChuanConfiguration(),
    "HalfWuChuanConfiguration\n(吴川半场 33×44m)":     HalfWuChuanConfiguration(),
    "QingYuanConfiguration\n(清远全场 58×36m)":        QingYuanConfiguration(),
    "HalfQingYuanConfiguration\n(清远半场 29×36m)":    HalfQingYuanConfiguration(),
    "CUMTBPitchConfiguration\n(CUMTB 105×68m)":        CUMTBPitchConfiguration(),
}

cols = 3
rows = (len(configs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
axes = axes.flatten()

for ax, (name, cfg) in zip(axes, configs.items()):
    img = draw_pitch_with_keypoints(cfg, scale=0.06)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(name, fontsize=10)
    ax.axis("off")
    info = f"长×宽: {cfg.length//100}m × {cfg.width//100}m\n关键点数: {len(cfg.vertices)}"
    ax.text(0.02, 0.02, info, transform=ax.transAxes, fontsize=8, color="white",
            bbox=dict(facecolor="black", alpha=0.55, pad=3))

for ax in axes[len(configs):]:
    ax.axis("off")

plt.suptitle("所有球场配置一览", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 单个配置详细查看

修改下方 `cfg` 变量即可切换任意已有配置，或粘入自定义配置类后直接实例化。

In [ ]:
import matplotlib.pyplot as plt
import cv2
from tools.annotation.pitch_annotation_tool import draw_pitch_with_keypoints

# ── 修改这里来切换配置 ──────────────────────────────────────────────────────
cfg = HalfWuChuanConfiguration()
# ────────────────────────────────────────────────────────────────────────────

img = draw_pitch_with_keypoints(cfg, scale=0.12)
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(
    f"{type(cfg).__name__}  |  长×宽: {cfg.length//100}m × {cfg.width//100}m  |  关键点数: {len(cfg.vertices)}",
    fontsize=12,
)
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"vertices : {len(cfg.vertices)}")
print(f"labels   : {len(cfg.labels)}")
print(f"colors   : {len(cfg.colors)}")
print(f"edges    : {len(cfg.edges)}")

### 自定义球场配置示例

以下是一个最小可运行的自定义半场配置。参照 README 第六章中的说明，修改尺寸参数、补全 `vertices` / `edges` / `labels` / `colors` 后，运行下方单元格即可立即验证。

In [ ]:
import matplotlib.pyplot as plt
import cv2
from dataclasses import dataclass, field
from typing import List, Tuple
from tools.annotation.pitch_annotation_tool import draw_pitch_with_keypoints


@dataclass
class CustomHalfPitch:
    """自定义半场示例（30m × 40m）。修改下方参数后重新运行即可。"""
    width: int = 3000               # [cm] 场地宽度
    length: int = 4000              # [cm] 场地长度
    penalty_box_width: int = 1800   # [cm] 大禁区宽
    penalty_box_length: int = 800   # [cm] 大禁区纵深
    goal_box_width: int = 1000      # [cm] 小禁区宽
    goal_box_length: int = 300      # [cm] 小禁区纵深
    centre_circle_radius: int = 500 # [cm] 中圈半径
    penalty_spot_distance: int = 600 # [cm] 点球点到底线距离

    @property
    def vertices(self) -> List[Tuple[int, int]]:
        return [
            (0, 0),                                                           # 1. 左上角
            (0, (self.width - self.penalty_box_width) / 2),                  # 2. 大禁区上
            (0, (self.width - self.goal_box_width) / 2),                     # 3. 小禁区上
            (0, (self.width + self.goal_box_width) / 2),                     # 4. 小禁区下
            (0, (self.width + self.penalty_box_width) / 2),                  # 5. 大禁区下
            (0, self.width),                                                  # 6. 左下角
            (self.goal_box_length, (self.width - self.goal_box_width) / 2),  # 7. 小禁区右上
            (self.goal_box_length, (self.width + self.goal_box_width) / 2),  # 8. 小禁区右下
            (self.penalty_spot_distance, self.width / 2),                    # 9. 点球点
            (self.penalty_box_length, (self.width - self.penalty_box_width) / 2),  # 10. 大禁区右上
            (self.penalty_box_length, (self.width + self.penalty_box_width) / 2),  # 11. 大禁区右下
            (self.length, 0),                                                 # 12. 中线上端
            (self.length, self.width / 2 - self.centre_circle_radius),       # 13. 中圈上端
            (self.length, self.width / 2 + self.centre_circle_radius),       # 14. 中圈下端
            (self.length, self.width),                                        # 15. 中线下端
            (self.length - self.centre_circle_radius, self.width / 2),       # 16. 中圈左点
            (self.length, self.width / 2),                                    # 17. 中点
            (0, self.width / 2),                                              # 18. 底线中点
        ]

    edges: List[Tuple[int, int]] = field(default_factory=lambda: [
        (1, 2), (2, 3), (3, 4), (4, 5), (5, 6),   # 左底线
        (3, 7), (7, 8), (8, 4),                     # 小禁区
        (2, 10), (10, 11), (11, 5),                 # 大禁区
        (1, 12), (6, 15),                           # 上下边线
        (12, 13), (13, 17), (17, 14), (14, 15),    # 中线
    ])

    labels: List[str] = field(default_factory=lambda: [
        "01", "02", "03", "04", "05", "06",
        "07", "08", "09", "10", "11",
        "12", "13", "14", "15", "16", "17", "18",
    ])

    colors: List[str] = field(default_factory=lambda:
        ["#FF1493"] * 11 + ["#00BFFF"] * 6 + ["#FFFFFF"] * 1
    )


# ── 一致性检查 ────────────────────────────────────────────────────────────
cfg = CustomHalfPitch()
assert len(cfg.vertices) == len(cfg.labels) == len(cfg.colors), (
    f"长度不一致: vertices={len(cfg.vertices)}, "
    f"labels={len(cfg.labels)}, colors={len(cfg.colors)}"
)
print(f"✅ 一致性检查通过  vertices={len(cfg.vertices)}  labels={len(cfg.labels)}  colors={len(cfg.colors)}")

# ── 可视化 ────────────────────────────────────────────────────────────────
img = draw_pitch_with_keypoints(cfg, scale=0.13)
plt.figure(figsize=(11, 7))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(
    f"{type(cfg).__name__}  |  长×宽: {cfg.length//100}m × {cfg.width//100}m  |  关键点数: {len(cfg.vertices)}",
    fontsize=12,
)
plt.axis("off")
plt.tight_layout()
plt.show()